In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
import joblib
import os

# -------------------- Create folder for outputs --------------------
os.makedirs("outputs", exist_ok=True)

# -------------------- 1. Load Dataset --------------------
data = pd.read_csv("students.csv")
data.columns = data.columns.str.strip().str.lower()

# -------------------- 2. Create Target --------------------
def map_dropout(text):
    text = str(text).lower()
    if "high" in text or "risk" in text:
        return 1
    return 0

data["dropout_risk"] = data["remarks"].apply(map_dropout)

# -------------------- 3. Encode Categorical --------------------
data["gender"] = data["gender"].map({"male": 0, "female": 1})
data["income"] = data["income"].map({"low": 0, "medium": 1, "high": 2})

# -------------------- 4. Drop unnecessary --------------------
data = data.drop(columns=["name", "remarks"])

# -------------------- 5. Features --------------------
FEATURES = ["attendance", "marks", "age", "gender", "income"]
X = data[FEATURES]
y = data["dropout_risk"]

# -------------------- 6. Train-Test Split --------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------- 7. Train Model --------------------
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# -------------------- 8. Predictions --------------------
y_pred = model.predict(X_test)

# -------------------- 9. Evaluation --------------------
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------- 10. Feature Importance Plot --------------------
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure()
plt.bar(importance["Feature"], importance["Importance"])
plt.xticks(rotation=45)
plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png")
plt.close()

# -------------------- 11. Confusion Matrix Plot --------------------
cm = confusion_matrix(y_test, y_pred)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png")
plt.close()

# -------------------- 12. Logistic Regression Curve --------------------
X_att = X_test[["attendance"]]

log_model = LogisticRegression()
log_model.fit(X_att, y_test)

X_range = np.linspace(X_att.min().values[0], X_att.max().values[0], 100).reshape(-1, 1)
y_prob = log_model.predict_proba(X_range)[:, 1]

plt.figure()
plt.scatter(X_att, y_test, label="Actual Data")
plt.plot(X_range, y_prob, linewidth=2, label="Logistic Curve")

plt.xlabel("Attendance")
plt.ylabel("Dropout Probability")
plt.title("Attendance vs Dropout Risk")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/logistic_curve.png")
plt.close()

# -------------------- 13. Save Model --------------------
joblib.dump(model, "dropout_model.pkl")

print("\n✅ Model saved as dropout_model.pkl")
print("📊 Graphs saved in /outputs folder")

KeyError: 'income_level'